# End-to-end encrypted inference: provable, not promised

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/08-e2ee-encryption.ipynb)

_Sabrina Aquino, Venice AI / Base Batches 003_

Most AI providers ask you to trust them. Venice's E2EE mode encrypts your prompt on the client with a key Venice never sees, decrypts it only inside a Trusted Execution Environment, and gives you a cryptographic attestation that proves the enclave was honest. We will implement it from scratch and inspect every byte.

## What you will build

1. **A privacy mode comparison table** so you know what each tier actually guarantees.
2. **The full E2EE handshake** end to end: fetch the enclave's signed key, derive a shared secret with ECDH (secp256k1), encrypt the prompt with AES-256-GCM, and verify the attestation.
3. **A side-by-side diff** of what Venice would see in private mode vs E2EE mode for the same prompt.
4. **Decrypt the response** the model produced inside the enclave.

Cost: zero extra. E2EE is included on Pro tiers and bundled with TEE.

## Privacy modes at a glance

| Mode | Who can read your prompt | Hardware proof | Default |
|---|---|---|---|
| Anonymized (3rd party) | 3rd party provider only, never linked to you | no | for proxied frontier models |
| Private (Venice default) | Venice infrastructure for the duration of inference, then discarded | no | yes, for open-source models |
| TEE | Only the verified enclave. Even Venice operators cannot read it | yes (remote attestation) | opt-in |
| E2EE | Only the verified enclave. Encrypted on your device first. Even Venice's network cannot see plaintext | yes (attestation + ECDH) | opt-in |

The whole point of E2EE: you do not have to trust Venice. You verify.

## Setup

In [ ]:
%pip install --quiet openai requests python-dotenv rich coincurve cryptography

In [ ]:
import os, time, json
from textwrap import shorten

# Pick up the API key from Colab secrets, environment, or interactive prompt
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(api_key=api_key, base_url="https://api.venice.ai/api/v1")
print("Connected to Venice")

In [ ]:
import json, base64, hashlib, requests
import coincurve
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

API_BASE = "https://api.venice.ai/api/v1"
HEADERS = {"Authorization": f"Bearer {api_key}"}

def b64e(b: bytes) -> str:
    return base64.b64encode(b).decode()

def b64d(s: str) -> bytes:
    return base64.b64decode(s)

## 1. Fetch the enclave's signed public key

Venice rotates a per-enclave keypair. The public half is published with a remote attestation that proves it came from a real TEE (Intel TDX / NVIDIA Confidential Computing). You should verify the attestation against your expected enclave measurement, but for the demo we just inspect it.

In [ ]:
r = requests.get(f"{API_BASE}/keys/e2ee", headers=HEADERS, timeout=30)
key_info = r.json()
print(json.dumps(key_info, indent=2)[:600], "...")
enclave_pub_hex = key_info.get("public_key") or key_info.get("publicKey")
attestation    = key_info.get("attestation")
print("\nEnclave public key (compressed secp256k1):", enclave_pub_hex)

## 2. Generate your ephemeral keypair and derive the shared secret

ECDH over secp256k1 (the same curve Bitcoin and Ethereum use). Your private key never leaves your machine. The enclave private key never leaves the enclave. Only the resulting shared secret can decrypt the prompt.

In [ ]:
my_priv = coincurve.PrivateKey()
my_pub  = my_priv.public_key.format(compressed=True)
print("My ephemeral public key:", my_pub.hex())

enclave_pub = coincurve.PublicKey(bytes.fromhex(enclave_pub_hex))
shared = my_priv.ecdh(enclave_pub.format(compressed=True))
session_key = hashlib.sha256(shared).digest()
print("Derived 256-bit AES key:", session_key.hex()[:32], "...")

## 3. Encrypt the prompt with AES-256-GCM

GCM gives us confidentiality and authentication in one shot. Without the right key the ciphertext is indistinguishable from random bytes.

In [ ]:
import os as _os

PROMPT = (
    "I am a doctor reviewing a patient case. The patient is a 47-year-old female with "
    "chest pain radiating to the left arm. Suggest a differential diagnosis."
)

aesgcm = AESGCM(session_key)
nonce  = _os.urandom(12)
ciphertext = aesgcm.encrypt(nonce, PROMPT.encode(), None)

envelope = {
    "client_pub": my_pub.hex(),
    "nonce":      b64e(nonce),
    "ciphertext": b64e(ciphertext),
}
print("Ciphertext size:", len(ciphertext), "bytes")
print("Ciphertext (first 60 bytes):", b64e(ciphertext)[:60], "...")
print()
print("Envelope keys:", list(envelope.keys()))

## 4. The diff: what Venice sees in Private mode vs E2EE mode

This is the punchline. In Private mode the request body contains the literal prompt. In E2EE mode it contains an encrypted blob. Same outcome for the user, completely different threat model for the operator.

In [ ]:
import pandas as pd

private_request = {
    "model": "llama-3.3-70b",
    "messages": [{"role": "user", "content": PROMPT}],
}

e2ee_request = {
    "model": "llama-3.3-70b",
    "encrypted": envelope,
}

pd.DataFrame({
    "mode":         ["Private (default)", "E2EE"],
    "request_body": [json.dumps(private_request)[:200], json.dumps(e2ee_request)[:200]],
})

## 5. Send the encrypted request and decrypt the reply

The enclave decrypts your prompt, runs the model, encrypts the response with the same shared key, and returns it. Plaintext never crosses the network.

In [ ]:
r = requests.post(
    f"{API_BASE}/chat/completions",
    headers={**HEADERS, "Content-Type": "application/json"},
    json={
        "model": "llama-3.3-70b",
        "encrypted": envelope,
    },
    timeout=120,
)
print("Status:", r.status_code)
encrypted_resp = r.json()
print("Encrypted response keys:", list(encrypted_resp.keys())[:6])

In [ ]:
nonce_resp  = b64d(encrypted_resp["nonce"])
cipher_resp = b64d(encrypted_resp["ciphertext"])

plaintext = aesgcm.decrypt(nonce_resp, cipher_resp, None).decode()
parsed = json.loads(plaintext)
print(parsed["choices"][0]["message"]["content"])

## 6. Verify the attestation

The most important step. The attestation is a signed quote from the TEE hardware that says "the code running here has measurement X, and this public key belongs to me." You should compare the measurement against the published Venice build hash. If they match, the enclave is honest. If they do not, abort.

For the workshop we just print the attestation. In production you would parse the quote with a verifier like [Phala's attestation explorer](https://proof.t16z.com/) or the [Intel TDX SDK](https://github.com/intel/SGXDataCenterAttestationPrimitives).

In [ ]:
import textwrap
print("Attestation type:", type(attestation).__name__)
preview = json.dumps(attestation, indent=2) if isinstance(attestation, dict) else str(attestation)
print(textwrap.shorten(preview, width=600))

## What we just proved

1. Your prompt was encrypted on your laptop using a key Venice never had.
2. Only an enclave holding the matching private key could decrypt it.
3. The enclave's identity is provable via a hardware-signed attestation.
4. The response came back encrypted to the same session key.

If the enclave is compromised, the attestation breaks and you abort. If Venice's operators are subpoenaed, the only thing they can hand over is ciphertext. That is the difference between *trust us* and *here is your receipt*.

Welcome to provable AI privacy. Now go ship something with it.